In [21]:
import pandas as pd
import numpy as np

from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

In [22]:
project_root = Path.cwd().parent

model_dir = project_root / "models"

recommendation_dir = (
    project_root
    / "data"
    / "processed"
    / "recommendation"
)

In [24]:
import os

os.listdir(recommendation_dir)

[' reviews_processed_popularity.csv', 'products_processed_popularity.csv']

In [26]:
products = pd.read_csv(
    recommendation_dir / "products_processed_popularity.csv",
    low_memory=False
)

reviews = pd.read_csv(
    recommendation_dir / " reviews_processed_popularity.csv",
    low_memory=False
)

In [ ]:
# Content-Based
tfidf = joblib.load(
    model_dir / "recommendation/content_based/tfidf_vectorizer.pkl"
)

content_similarity = joblib.load(
    model_dir / "recommendation/content_based/cosine_similarity.pkl"
)

content_products = joblib.load(
    model_dir / "recommendation/content_based/product_indices.pkl"
)

# Collaborative
item_knn = joblib.load(
    model_dir / "recommendation/collab/item_knn.pkl"
)

item_user_matrix = joblib.load(
    model_dir / "recommendation/collab/item_user_matrix.pkl"
)



# Hybrid
hybrid_products = joblib.load(
    model_dir / "recommendation/hybrid/hybrid_products.pkl"
)

hybrid_config = joblib.load(
    model_dir / "recommendation/hybrid/hybrid_config.pkl"
)

In [74]:
cf_products = pd.read_csv(
    model_dir / "recommendation/collab/product_map.csv"
)


cf_products = cf_products.merge(
    products[
        [
            "product_id",
            "price_usd"
        ]
    ],
    on="product_id",
    how="left"
)


cf_products.head()

,product_idx,product_id,product_name,brand_name,primary_category,secondary_category,price_usd
0,199,P394639,The True Cream Aqua Bomb,belif,Skincare,Moisturizers,38.0
1,236,P4016,Acne Control Clarifying Cleanser,Murad,Skincare,Cleansers,36.0
2,467,P427415,100% Organic Cold-Pressed Rose Hip Seed Oil,The Ordinary,Skincare,Moisturizers,10.9
3,488,P428250,Oil-Absorbing Pore Treatment Strips,Peace Out,Skincare,Treatments,19.0
4,673,P440651,Mini Acne Control Clarifying Cleanser,Murad,Skincare,Mini Size,14.0


In [28]:
type(content_similarity)

numpy.ndarray

In [29]:
type(item_user_matrix)

scipy.sparse._csc.csc_matrix

In [30]:
hybrid_config

{'content_weight': 0.5, 'collab_weight': 0.3, 'popularity_weight': 0.2}

In [66]:
def recommend_content(product_id, top_n=10):

    if product_id not in content_products["product_id"].values:
        return pd.DataFrame()

    product_idx = (
        content_products[
            content_products["product_id"] == product_id
        ]
        ["product_idx"]
        .values[0]
    )

    similarity_scores = content_similarity[product_idx]

    similar_indices = (
        np.argsort(similarity_scores)[::-1]
    )

    recommendations = []

    for idx in similar_indices:

        if idx == product_idx:
            continue

        recommendations.append(
            {
                "product_id": content_products.iloc[idx]["product_id"],
                "content_score": similarity_scores[idx]
            }
        )

        if len(recommendations) == top_n:
            break


    result = pd.DataFrame(recommendations)


    result = result.merge(
        hybrid_products[
            [
                "product_id",
                "product_name",
                "brand_name",
                "primary_category",
                "price_usd"
            ]
        ],
        on="product_id",
        how="left"
    )


    return result[
        [
            "product_id",
            "product_name",
            "brand_name",
            "primary_category",
            "price_usd",
            "content_score"
        ]
    ]

In [75]:
cf_product_to_idx = dict(
    zip(
        cf_products["product_id"],
        cf_products["product_idx"]
    )
)


def recommend_collaborative(product_id, top_n=10):

    if product_id not in cf_product_to_idx:
        return pd.DataFrame()


    product_idx = cf_product_to_idx[product_id]


    distances, indices = item_knn.kneighbors(
        item_user_matrix[product_idx],
        n_neighbors=top_n+1
    )


    recommendations=[]


    for distance, idx in zip(
        distances[0][1:],
        indices[0][1:]
    ):

        similarity = 1-distance

        recommendations.append(
            {
                "product_idx": idx,
                "collab_score": similarity
            }
        )


    result = pd.DataFrame(recommendations)


    result = result.merge(
        cf_products,
        on="product_idx",
        how="left"
    )


    return result[
        [
            "product_id",
            "product_name",
            "brand_name",
            "primary_category",
            "price_usd",
            "collab_score"
        ]
    ].sort_values(
        "collab_score",
        ascending=False
    )

In [76]:
def hybrid_recommend(product_id, top_n=10):

    content = recommend_content(
        product_id,
        top_n=50
    )


    collab = recommend_collaborative(
        product_id,
        top_n=50
    )


    hybrid = (
        content[
            [
                "product_id",
                "content_score"
            ]
        ]
        .merge(
            collab[
                [
                    "product_id",
                    "collab_score"
                ]
            ],
            on="product_id",
            how="outer"
        )
    )


    hybrid["content_score"] = (
        hybrid["content_score"]
        .fillna(0)
    )


    hybrid["collab_score"] = (
        hybrid["collab_score"]
        .fillna(0)
    )


    popularity = hybrid_products[
        [
            "product_id",
            "popularity_norm"
        ]
    ]


    hybrid = hybrid.merge(
        popularity,
        on="product_id",
        how="left"
    )


    hybrid["popularity_norm"] = (
        hybrid["popularity_norm"]
        .fillna(0)
    )


    w_content = hybrid_config["content_weight"]
    w_collab = hybrid_config["collab_weight"]
    w_pop = hybrid_config["popularity_weight"]


    hybrid["final_score"] = (
        w_content * hybrid["content_score"]
        +
        w_collab * hybrid["collab_score"]
        +
        w_pop * hybrid["popularity_norm"]
    )


    hybrid = hybrid.merge(
        hybrid_products[
            [
                "product_id",
                "product_name",
                "brand_name",
                "primary_category",
                "price_usd"
            ]
        ],
        on="product_id",
        how="left"
    )


    return hybrid[
        [
            "product_id",
            "product_name",
            "brand_name",
            "primary_category",
            "price_usd",
            "final_score",
            "content_score",
            "collab_score",
            "popularity_norm"
        ]
    ].sort_values(
        "final_score",
        ascending=False
    ).head(top_n)

In [31]:
reviews.columns.tolist()

['author_id',
 'rating',
 'is_recommended',
 'helpfulness',
 'total_feedback_count',
 'total_neg_feedback_count',
 'total_pos_feedback_count',
 'submission_time',
 'review_text',
 'review_title',
 'skin_tone',
 'eye_color',
 'skin_type',
 'hair_color',
 'product_id',
 'product_name',
 'brand_name',
 'price_usd',
 'positive_signal',
 'engagement_score',
 'review_length',
 'recency_days',
 'time_weight',
 'interaction_strength',
 'rating_score',
 'recommendation_score',
 'helpfulness_score',
 'interaction_score']

In [32]:
interactions = reviews[
    [
        "author_id",
        "product_id",
        "interaction_score"
    ]
].copy()

In [33]:
print("Users:", interactions["author_id"].nunique())
print("Products:", interactions["product_id"].nunique())
print("Interactions:", len(interactions))

interactions.head()

Users: 503216
Products: 2351
Interactions: 1088891


,author_id,product_id,interaction_score
0,538863,P420652,0.2
1,549704,P218700,1.0
2,557770,P232903,1.0
3,562130,P381030,1.0
4,582399,P420652,0.9


In [34]:
user_counts = (
    interactions
    .groupby("author_id")
    .size()
)

In [35]:
user_counts.describe()

count    503216.000000
mean          2.163864
std           3.392856
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         292.000000
dtype: float64

In [36]:
active_users = user_counts[
    user_counts >= 5
].index

In [37]:
evaluation_df = interactions[
    interactions.author_id.isin(active_users)
].copy()

In [38]:
print(evaluation_df.shape)

print(
    evaluation_df.author_id.nunique()
)

print(
    evaluation_df.product_id.nunique()
)

(370966, 3)
40435
2257


In [39]:
train_rows = []
test_rows = []

In [40]:
for user, group in evaluation_df.groupby("author_id"):

    if len(group) < 2:
        continue

    train, test = train_test_split(
        group,
        test_size=1,
        random_state=42
    )

    train_rows.append(train)
    test_rows.append(test)

In [41]:
train_df = pd.concat(
    train_rows,
    ignore_index=True
)

test_df = pd.concat(
    test_rows,
    ignore_index=True
)

In [42]:
print(train_df.shape)
print(test_df.shape)

(330531, 3)
(40435, 3)


In [43]:
print(train_df.head())

     author_id product_id  interaction_score
0  10000117144    P394639               0.90
1  10000117144    P453253               0.96
2  10000117144    P427415               0.90
3  10000117144    P440651               0.90
4  10000117144    P428250               1.00


In [44]:
print(test_df.head())

     author_id product_id  interaction_score
0  10000117144      P4016                0.9
1  10000217994    P456213                0.8
2  10000290716    P375849                0.9
3  10000770719    P477413                0.9
4  10000892274    P392235                0.9


In [45]:
print("Train users:", train_df["author_id"].nunique())
print("Test users :", test_df["author_id"].nunique())

print()

print("Train products:", train_df["product_id"].nunique())
print("Test products :", test_df["product_id"].nunique())

Train users: 40435
Test users : 40435

Train products: 2246
Test products : 1762


In [46]:
train_users = set(train_df["author_id"])
train_products = set(train_df["product_id"])

test_df = test_df[
    test_df["author_id"].isin(train_users)
    &
    test_df["product_id"].isin(train_products)
].copy()

print(test_df.shape)

(40421, 3)


In [47]:
ground_truth = (
    test_df
    .groupby("author_id")["product_id"]
    .apply(set)
    .to_dict()
)

print("Evaluation users:", len(ground_truth))

Evaluation users: 40421


In [48]:
next(iter(ground_truth.items()))

('10000117144', {'P4016'})

In [49]:
evaluation_users = list(ground_truth.keys())

print(len(evaluation_users))

evaluation_users[:5]

40421


['10000117144', '10000217994', '10000290716', '10000770719', '10000892274']

In [50]:
def precision_at_k(recommended, relevant, k=10):

    recommended = recommended[:k]

    if len(recommended) == 0:
        return 0

    hits = len(
        set(recommended).intersection(relevant)
    )

    return hits / k

In [51]:
recommended = ["A", "B", "C", "D", "E"]
relevant = {"B", "E", "F"}

precision_at_k(
    recommended,
    relevant,
    k=5
)

0.4

In [52]:
def recall_at_k(recommended, relevant, k=10):

    recommended = recommended[:k]

    if len(relevant) == 0:
        return 0

    hits = len(
        set(recommended).intersection(relevant)
    )

    return hits / len(relevant)

In [53]:
recall_at_k(
    recommended,
    relevant,
    k=5
)

0.6666666666666666

In [54]:
def hit_rate_at_k(recommended, relevant, k=10):

    recommended = recommended[:k]

    return int(
        len(
            set(recommended).intersection(relevant)
        ) > 0
    )

In [55]:
hit_rate_at_k(
    recommended,
    relevant,
    k=5
)

1

In [56]:
def average_precision_at_k(
    recommended,
    relevant,
    k=10
):

    recommended = recommended[:k]

    score = 0
    hits = 0

    for i, product in enumerate(recommended):

        if product in relevant:

            hits += 1

            score += hits / (i + 1)

    if len(relevant) == 0:
        return 0

    return score / min(len(relevant), k)

In [57]:
average_precision_at_k(
    recommended,
    relevant,
    k=5
)

0.3

In [58]:
import numpy as np


def ndcg_at_k(
    recommended,
    relevant,
    k=10
):

    recommended = recommended[:k]

    dcg = 0

    for i, product in enumerate(recommended):

        if product in relevant:

            dcg += 1 / np.log2(i + 2)

    ideal_hits = min(len(relevant), k)

    idcg = sum(
        1 / np.log2(i + 2)
        for i in range(ideal_hits)
    )

    if idcg == 0:
        return 0

    return dcg / idcg

In [59]:
ndcg_at_k(
    recommended,
    relevant,
    k=5
)

np.float64(0.4776237035032179)

In [60]:
recommended = [
    "A",
    "B",
    "C",
    "D",
    "E"
]

relevant = {
    "B",
    "E",
    "F"
}

print("Precision :", precision_at_k(recommended, relevant))
print("Recall    :", recall_at_k(recommended, relevant))
print("Hit Rate  :", hit_rate_at_k(recommended, relevant))
print("MAP       :", average_precision_at_k(recommended, relevant))
print("NDCG      :", ndcg_at_k(recommended, relevant))

Precision : 0.2
Recall    : 0.6666666666666666
Hit Rate  : 1
MAP       : 0.3
NDCG      : 0.4776237035032179


In [61]:
def evaluate_model(
    recommendation_function,
    users,
    ground_truth,
    k=10
):

    precisions = []
    recalls = []
    hit_rates = []
    maps = []
    ndcgs = []

    skipped_users = 0

    for user in users:

        try:

            recommendations = recommendation_function(
                user,
                top_n=k
            )

            if recommendations is None or len(recommendations) == 0:
                skipped_users += 1
                continue

            recommended_products = recommendations[
                "product_id"
            ].tolist()

            relevant_products = ground_truth[user]

            precisions.append(
                precision_at_k(
                    recommended_products,
                    relevant_products,
                    k
                )
            )

            recalls.append(
                recall_at_k(
                    recommended_products,
                    relevant_products,
                    k
                )
            )

            hit_rates.append(
                hit_rate_at_k(
                    recommended_products,
                    relevant_products,
                    k
                )
            )

            maps.append(
                average_precision_at_k(
                    recommended_products,
                    relevant_products,
                    k
                )
            )

            ndcgs.append(
                ndcg_at_k(
                    recommended_products,
                    relevant_products,
                    k
                )
            )

        except Exception:
            skipped_users += 1

    return {
        "Users Evaluated": len(precisions),
        "Users Skipped": skipped_users,
        "Precision@K": np.mean(precisions),
        "Recall@K": np.mean(recalls),
        "HitRate@K": np.mean(hit_rates),
        "MAP@K": np.mean(maps),
        "NDCG@K": np.mean(ndcgs)
    }

In [62]:
user_history = (
    train_df
    .groupby("author_id")["product_id"]
    .apply(list)
    .to_dict()
)

len(user_history)

40435

In [63]:
next(iter(user_history.items()))

('10000117144',
 ['P394639', 'P453253', 'P427415', 'P440651', 'P428250', 'P504794'])

In [64]:
def content_recommend_for_user(
    author_id,
    top_n=10
):

    if author_id not in user_history:
        return pd.DataFrame()

    seed_product = user_history[author_id][-1]

    return recommend_content(
        seed_product,
        top_n=top_n
    )

In [69]:
content_recommend_for_user(
    evaluation_users[0],
    top_n=5
)

,product_id,product_name,brand_name,primary_category,price_usd,content_score
0,P504773,Priming Moisturizer Rich Face Cream with Ceram...,Glossier,Skincare,35.0,0.310632
1,P503642,Squalane + Vitamin C Rose Brightening Moisturizer,Biossance,Skincare,56.0,0.251350
2,P504840,Hydrate + Purify Mini Bestsellers Discovery Sk...,innisfree,Skincare,18.0,0.244626
3,P501587,Deepwater Nourishing Lip Mask with Sea Moss an...,CAY SKIN,Skincare,22.0,0.240935
4,P505209,Barrier Culture Moisturizer with Niacinamide &...,The Nue Co.,Skincare,65.0,0.234855


In [70]:
def collaborative_recommend_for_user(
    author_id,
    top_n=10
):

    if author_id not in user_history:
        return pd.DataFrame()

    seed_product = user_history[author_id][-1]

    return recommend_collaborative(
        seed_product,
        top_n=top_n
    )

In [77]:
collaborative_recommend_for_user(
    evaluation_users[0],
    top_n=5
)

,product_id,product_name,brand_name,primary_category,price_usd,collab_score
0,P504524,Milky Jelly Gentle Gel Face Cleanser,Glossier,Skincare,19.0,0.205811
1,P504784,Priming Moisturizer Lightweight Buildable Face...,Glossier,Skincare,24.0,0.138989
2,P504790,Cleanser Concentrate AHA Clarifying and Exfoli...,Glossier,Skincare,21.0,0.105665
3,P504810,Priming Moisturizer Balance Oil-Control Gel-Cream,Glossier,Skincare,25.0,0.099304
4,P504523,Futuredew Facial Oil-Serum Hybrid,Glossier,Skincare,26.0,0.094964


In [72]:
def hybrid_recommend_for_user(
    author_id,
    top_n=10
):

    if author_id not in user_history:
        return pd.DataFrame()

    seed_product = user_history[author_id][-1]

    return hybrid_recommend(
        seed_product,
        top_n=top_n
    )

In [78]:
hybrid_recommend_for_user(
    evaluation_users[0],
    top_n=5
)

,product_id,product_name,brand_name,primary_category,price_usd,final_score,content_score,collab_score,popularity_norm
24,P455936,Lip Butter Balm,Summer Fridays,Skincare,24.0,0.284435,0.212282,0.000000,0.891470
82,P504773,Priming Moisturizer Rich Face Cream with Ceram...,Glossier,Skincare,35.0,0.272289,0.310632,0.000000,0.584863
53,P479330,Water Sleeping Mask with Squalane,LANEIGE,Skincare,32.0,0.263798,0.204247,0.012285,0.789946
19,P446930,Cream Skin Toner & Moisturizer,LANEIGE,Skincare,33.0,0.262209,0.186687,0.000000,0.844327
84,P504784,Priming Moisturizer Lightweight Buildable Face...,Glossier,Skincare,24.0,0.259973,0.196813,0.138989,0.599347


In [79]:
content_results = evaluate_model(
    recommendation_function=content_recommend_for_user,
    users=evaluation_users,
    ground_truth=ground_truth,
    k=10
)

content_results

{'Users Evaluated': 40421,
 'Users Skipped': 0,
 'Precision@K': np.float64(0.00860443828702902),
 'Recall@K': np.float64(0.08604438287029019),
 'HitRate@K': np.float64(0.08604438287029019),
 'MAP@K': np.float64(0.04809467654523442),
 'NDCG@K': np.float64(0.05699924390994288)}

In [80]:
collab_results = evaluate_model(
    recommendation_function=collaborative_recommend_for_user,
    users=evaluation_users,
    ground_truth=ground_truth,
    k=10
)

collab_results

{'Users Evaluated': 40370,
 'Users Skipped': 51,
 'Precision@K': np.float64(0.018887787961357445),
 'Recall@K': np.float64(0.18887787961357444),
 'HitRate@K': np.float64(0.18887787961357444),
 'MAP@K': np.float64(0.08659145955467482),
 'NDCG@K': np.float64(0.11086414059286456)}

In [ ]:
hybrid_results = evaluate_model(
    recommendation_function=hybrid_recommend_for_user,
    users=evaluation_users,
    ground_truth=ground_truth,
    k=10
)

hybrid_results

In [ ]:
evaluation_results = pd.DataFrame(
    [
        {
            "Model":"Content-Based",
            **content_results
        },
        {
            "Model":"Collaborative Filtering",
            **collab_results
        },
        {
            "Model":"Hybrid",
            **hybrid_results
        }
    ]
)


evaluation_results

In [ ]:
evaluation_results.sort_values(
    "NDCG@K",
    ascending=False
)